# Öğretmen model hız ölçümü · Gemma 4 E4B
**Amaç:** Öğretmen modelin Colab T4'te **saatte kaç hikâye ve kaç kart** yazdığını ölçmek. Bu sayıyla hedef veri miktarı ve gece başına üretim planlanır.

Yapılacak: hücreleri sırayla çalıştır (*Çalışma zamanı → Tümünü çalıştır*). Sonunda çıkan özet tabloyu `docs/raporlar/ogretmen_secimi.md`'ye yapıştır.

## 1 · Hazırlık
*Çalışma zamanı → Çalışma zamanı türünü değiştir → **T4 GPU*** seçili olmalı.

In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU YOK: T4 seç")

In [ ]:
# Drive: üretilen dosyalar buraya yazılır, Colab kopsa da kaybolmaz
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/ileri-programlama/02_Veri'
!mkdir -p "$DRIVE/uretilen" "$DRIVE/raporlar"

In [ ]:
# Özel depolar için salt okunur GitHub token'ı; URL'ye/dosyaya kaydedilmez.
# Colab çıktılarında veya paylaşılacak notebook hücrelerinde token yazmayın.
import base64
import getpass
import os
from pathlib import Path
import subprocess

token = getpass.getpass('GitHub token (özel AI ve Docs depolarına okuma erişimi): ')
git_env = os.environ.copy()
if token:
    yetki = base64.b64encode(('x-access-token:' + token).encode()).decode()
    git_env.update(GIT_CONFIG_COUNT='1',
                   GIT_CONFIG_KEY_0='http.https://github.com/.extraheader',
                   GIT_CONFIG_VALUE_0='Authorization: Basic ' + yetki)
    del yetki
git_env['GIT_TERMINAL_PROMPT'] = '0'
del token
try:
    for ad in ('AI', 'Docs'):
        adres = f'https://github.com/IleriProgramlamaProje/{ad}.git'
        hedef = Path('/content') / ad
        if not hedef.exists():
            subprocess.run(['git', 'clone', '--quiet', adres, str(hedef)], env=git_env, check=True)
        else:
            remote = subprocess.check_output(['git', '-C', str(hedef), 'remote', 'get-url', 'origin'], text=True).strip()
            if remote != adres:
                raise ValueError(f'{hedef}: beklenmeyen depo; klasörü kontrol edin')
            print(f'{ad}: mevcut kopya korunuyor; güncelleme gerekiyorsa önce yerel değişiklikleri kontrol edin.')
        print(ad, subprocess.check_output(['git', '-C', str(hedef), 'rev-parse', 'HEAD'], text=True).strip())
finally:
    git_env.clear()
os.chdir('/content/AI')
os.environ['PROJE_DOCS_DIR'] = '/content/Docs'


In [ ]:
# vLLM: toplu üretimde en hızlı motor (kurulum 3-5 dk). Kurulmazsa kod otomatik transformers'a düşer.
!pip install -q -U vllm transformers accelerate

In [ ]:
# Hugging Face girişi. Önce huggingface.co/google/gemma-4-E4B-it sayfasında lisansı kabul et.
from huggingface_hub import login
login()

## 2 · Ölçüm
Her tür için küçük bir üretim yapılır. İlk grup modeli ısıttığı için biraz yavaş olabilir; asıl sayı sonraki gruplardır.

In [ ]:
import sys; sys.path.insert(0, '/content/AI')
from ai.veri_uretimi.uretici import uret
from datetime import datetime
OLCUM = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
uret('hikaye', 128, f'{DRIVE}/uretilen/hiz_hikaye_{OLCUM}.jsonl', motor_adi='auto', grup=8, tohum=1)

In [ ]:
uret('kart', 128, f'{DRIVE}/uretilen/hiz_kart_{OLCUM}.jsonl', motor_adi='auto', grup=8, tohum=2)

## 3 · Özet

In [ ]:
import json, statistics
def ozet(yol, tur):
    K = [json.loads(s) for s in open(yol, encoding='utf-8')]
    sn = sum(k['saniye'] for k in K)           # kayıt başına süre toplamı = gerçek süre
    tok = [k['token'] for k in K]
    kelime = [len(k['cikti'].split()) for k in K]
    saat = len(K) / sn * 3600
    return {'tür': tur, 'kayıt': len(K), 'saatte': round(saat), '8 saatlik gecede': round(saat*8),
            'ort. token': round(statistics.mean(tok)), 'ort. kelime': round(statistics.mean(kelime))}
tablo = [ozet(f'{DRIVE}/uretilen/hiz_hikaye_{OLCUM}.jsonl','hikaye'), ozet(f'{DRIVE}/uretilen/hiz_kart_{OLCUM}.jsonl','kart')]
for t in tablo: print(t)
md = '| Tür | Saatte | 8 saatlik gecede | Ort. kelime |\n| --- | --- | --- | --- |\n' + '\n'.join(
    f"| {t['tür']} | {t['saatte']} | {t['8 saatlik gecede']} | {t['ort. kelime']} |" for t in tablo)
print(md); open(f'{DRIVE}/raporlar/hiz_olcumu_{OLCUM}.md','w', encoding='utf-8').write(md)

In [ ]:
# Göz atmak için birkaç örnek
for yol in ['hiz_hikaye','hiz_kart']:
    K = [json.loads(s) for s in open(f'{DRIVE}/uretilen/{yol}_{OLCUM}.jsonl', encoding='utf-8')]
    for k in K[:3]: print(k['malzeme'], '\n', k['cikti'], '\n' + '-'*60)

## 4 · Sonuç ne olacak?
- Tabloyu `docs/raporlar/ogretmen_secimi.md`'ye ekle.
- **Hedef veri miktarı** bu sayıya göre belirlenir (yol haritası 9. bölüm, 3. madde). Kabaca: 30M'lik model için birkaç yüz bin hikâye iyi bir hedef; saatte X hikâye × gece başına 8 saat × 2 hesap = günlük üretim.
- Örneklerde kurala uymayan çok şey varsa prompt'u (`ai/veri_uretimi/promptlar/*.txt`) düzeltip ölçümü tekrarla.